# Persistence XMY — Warming Level approach deconstruction

Pulls apart `xmy.generate_xmy()` for the WL=1.5, q=0.9, KLAX configuration so each step can be inspected.

Original call (do **not** run; we deconstruct it below):

```python
xmy = persistence_XMY(
    station_name="Los Angeles International Airport (KLAX)",
    q=0.9,
    warming_level=1.5,
    verbose=True,
)
xmy.generate_xmy()
```

`generate_xmy()` is just:

```python
self.load_all_variables()   # fetch + derive hourly + resample to daily
self.get_candidate_hours()  # → set_top_hours → persistence_get_top_hours
self.run_xmy_analysis()     # _make_8760_tables + smoothing
self.export_xmy_data()      # write EPW
```

We expand each one and stop at the failing reshape so we can inspect intermediate state.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import xarray as xr

from climakitae.explore.extreme_meteorological_year import (
    persistence_XMY,
    persistence_get_top_hours,
)
from climakitae.core.constants import UNSET

In [2]:
# Construct the object — runs __init__ only (no data fetched yet).
xmy = persistence_XMY(
    station_name="Los Angeles International Airport (KLAX)",
    q=0.9,
    warming_level=1.5,
    verbose=True,
)

# Configuration sanity check
print("q              :", xmy.q)
print("warming_level  :", xmy.warming_level)
print("start_year     :", xmy.start_year)
print("end_year       :", xmy.end_year)
print("stn_name       :", xmy.stn_name)
print("stn_lat,lon    :", xmy.stn_lat, xmy.stn_lon)
print("_skip_last     :", xmy._skip_last)
print("simulations   :", xmy.simulations)
print("UTC offset (h) :", xmy._get_utc_offset_hours())

Initializing persistence XMY object for Los Angeles International Airport (KLAX).
Generating p90 persistence XMY.
q              : 0.9
warming_level  : 1.5
start_year     : UNSET
end_year       : UNSET
stn_name       : Los Angeles International Airport (KLAX)
stn_lat,lon    : 33.93816 -118.3866
_skip_last     : True
simulations   : ['WRF_EC-Earth3_r1i1p1f1', 'WRF_MPI-ESM1-2-HR_r3i1p1f1', 'WRF_TaiESM1_r1i1p1f1', 'WRF_MIROC6_r1i1p1f1']
UTC offset (h) : -8


## 1. `load_all_variables()` — fetch raw + derive

This is the slow step (downloads and computes hourly variables for 4 simulations). Internally it calls `_fetch_raw_variable()` per variable, then merges, derives RH/dew point/wind, computes daily aggregates, and stores:

- `self._hourly_data` — hourly dataset used downstream
- `self.all_vars` — daily aggregates
- `self.air_temp_var` — hourly air temperature DataArray (the input to `persistence_get_top_hours`)

In [3]:
xmy.load_all_variables()

Loading data from catalog via ClimateData.
  Getting t2 (sets year range)...
2026-06-12 13:47:22 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

  Loading hourly variables in parallel...
  Fetching q2 (1hr)...
  Fetching psfc (1hr)...
  Fetching u10 (1hr)...
  Fetching v10 (1hr)...
2026-06-12 13:48:14 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

2026-06-12 13:48:14 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 


IOStream.flush timed out


[########################################] | 100% Completed | 16m 51s
  Resampling hourly data to daily statistics...
  Daily statistics ready.


In [4]:
# Inspect what landed on the object
print("hourly dims  :", dict(xmy._hourly_data.sizes))
print("hourly vars  :", list(xmy._hourly_data.data_vars))
print("air_temp_var :", type(xmy.air_temp_var).__name__, xmy.air_temp_var.sizes)
print()
print("start_year (post-load):", xmy.start_year)
print("end_year   (post-load):", xmy.end_year)

hourly dims  : {'simulation': 4, 'time': 262792}
hourly vars  : ['Air Temperature at 2m', 'Dew point temperature', 'Relative humidity', 'Wind speed at 10m', 'Wind direction at 10m', 'Water Vapor Mixing Ratio at 2m', 'Instantaneous downwelling shortwave flux at bottom', 'Shortwave surface downward direct normal irradiance', 'Shortwave surface downward diffuse irradiance', 'Instantaneous downwelling longwave flux at bottom', 'Surface Pressure']
air_temp_var : DataArray Frozen({'simulation': 4, 'time': 262792})

start_year (post-load): 2000
end_year   (post-load): 2029


### 1a. Diagnose the time axis of `air_temp_var`

This is where the 8752 vs 8760 mismatch lives. After `_fetch_raw_variable`:

- `add_dummy_time_to_wl` produces an exact `N × 8760` UTC dummy timeline starting `2000-01-01 00:00` (leap days filtered).
- The pipeline then does `data["time"] += pd.Timedelta(hours=offset_hours)` (KLAX → −8h) and `.sel(time=slice(start_year-01-01, end_year-12-31))`.
- Result for KLAX/PST: `30*8760 - 8 = 262 792` stamps; the **trailing** local-time year has only **8752** stamps.

In [5]:
da = xmy.air_temp_var
times = pd.DatetimeIndex(da.time.values)

print("total stamps           :", len(times))
print("total / 8760           :", len(times) / 8760)
print("floor(total / 8760)    :", len(times) // 8760)
print("ceil(total / 8760)     :", -(-len(times) // 8760))
print("first stamp            :", times[0])
print("last stamp             :", times[-1])
print()
print("per-year hour counts (first/last 5):")
yc = pd.Series(times.year).value_counts().sort_index()
print(yc.head())
print("...")
print(yc.tail())
print()
print("years with != 8760 stamps:")
print(yc[yc != 8760])

total stamps           : 262792
total / 8760           : 29.999086757990867
floor(total / 8760)    : 29
ceil(total / 8760)     : 30
first stamp            : 2000-01-01 00:00:00
last stamp             : 2029-12-31 15:00:00

per-year hour counts (first/last 5):
2000    8760
2001    8760
2002    8760
2003    8760
2004    8760
Name: count, dtype: int64
...
2025    8760
2026    8760
2027    8760
2028    8760
2029    8752
Name: count, dtype: int64

years with != 8760 stamps:
2029    8752
Name: count, dtype: int64


## 2. `get_candidate_hours()` — the failing step

Calls `set_top_hours()`, which calls:

```python
self.top_hours = persistence_get_top_hours(self.air_temp_var, self.q, skip_last=self._skip_last)
```

We inline the function below so we can stop at the failing reshape and inspect.

In [6]:
# Run the function as the code currently has it.
# If it raises (e.g. assign_coords length mismatch), we'll dive in below.
try:
    top_hours = persistence_get_top_hours(
        xmy.air_temp_var, xmy.q, skip_last=xmy._skip_last
    )
    print("OK, top_hours shape:", top_hours.shape)
    display(top_hours.head())
except Exception as e:
    print(f"{type(e).__name__}: {e}")

      📊 Processing 262,800 hours (30 years) of data
      🎯 Computing 90th percentile for each hour of year


KeyboardInterrupt: 

### 2a. Manual deconstruction of `persistence_get_top_hours`

Walk through the function body line-by-line so we can see exactly where the lengths diverge.

In [ ]:
data = xmy.air_temp_var
q = xmy.q
skip_last = xmy._skip_last

has_simulation = "simulation" in data.dims
simulations = data.simulation.values if has_simulation else [None]

print("has_simulation :", has_simulation)
print("simulations    :", list(simulations))
print("len(data.time) :", len(data.time))
print("skip_last      :", skip_last)

In [ ]:
# `skip_last` branch — this is what the user wants to fix.
#
# Goal stated by user: drop only the last MONTH (not the last full year)
# of the trailing partial year, so that we keep year N as a quasi-full year.
#
# Note: the existing code uses .loc[dict(year=..., month=...)] = np.inf, which
# requires `year` and `month` to be index dims — they are not on this DataArray
# (only `time` is). That's why it currently fails. Below we reproduce the
# *intent* with a month-aware mask on the time coordinate.

times = pd.DatetimeIndex(data.time.values)
last_year = int(times[-1].year)
last_month = int(times[-1].month)
print(f"last (year, month) of data: ({last_year}, {last_month})")

if skip_last:
    # WL+local-time leaves the trailing year |offset_hours| short of 8760
    # (e.g. KLAX/PST: last year ends Dec 31 15:00). Pad the tail back to a
    # clean N*8760 cycle, then NaN-mask the trailing month so it can't be
    # selected. NaN (not inf) — nanquantile/nanargmin both skip NaN.
    times = pd.DatetimeIndex(data.time.values)
    n_full_years = -(-len(times) // 8760)  # ceiling division
    missing = n_full_years * 8760 - len(times)

    da_work = data.copy(deep=True)
    if missing > 0:
        pad_start = times[-1] + pd.Timedelta(hours=1)
        pad_times = pd.date_range(start=pad_start, periods=missing, freq="h")
        pad_shape = list(data.shape)
        time_axis = data.dims.index("time")
        pad_shape[time_axis] = len(pad_times)
        pad_coords = {d: data.coords[d] for d in data.dims if d != "time"}
        pad_coords["time"] = pad_times
        pad = xr.DataArray(np.full(pad_shape, np.nan), dims=data.dims, coords=pad_coords)
        da_work = xr.concat([da_work, pad], dim="time")

    # NaN-mask the entire trailing local-time month
    last_t = pd.DatetimeIndex(da_work.time.values)[-1]
    last_year, last_month = int(last_t.year), int(last_t.month)
    mask = (da_work.time.dt.year == last_year) & (da_work.time.dt.month == last_month)
    da_work = da_work.where(~mask)
else:
    da_work = data

print("len(da_work.time)            :", len(da_work.time))
print("len(da_work.time) % 8760     :", len(da_work.time) % 8760)
print("len(da_work.time) / 8760     :", len(da_work.time) / 8760)

**Important:** dropping only the last month does **not** make the total a multiple of 8760. It leaves a trailing partial year (e.g. for KLAX/PST: `30*8760 − 8 − hours_in_last_month`). The downstream `np.tile(np.arange(1, 8761), n_years)[:total_hours]` and the `assign_coords(hour_of_year=...)` step assume each calendar position lines up with a fixed hour-of-year slot, which only works if data starts at Jan 1, hour 0 *and* every year is exactly 8760 long.

So if the goal is *only* drop the last month (not the whole partial year), the reshape logic in the loop also has to change — it can no longer use `(n_total // 8760) * 8760` as the usable slice, because that throws away whole years from the head if the partial year is at the tail. See section 3 for what the loop currently does and where it diverges.

In [ ]:
# What the current code does next (intentionally fragile, for inspection).
hours_per_year = 8760
total_hours = len(da_work.time)
n_years = total_hours // hours_per_year

hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]

print("total_hours          :", total_hours)
print("n_years (floor)      :", n_years)
print("len(hour_of_year_all):", len(hour_of_year_all))
print("⇒ assign_coords mismatch:", total_hours - len(hour_of_year_all))

In [ ]:
# Confirm: this is the call that raises if the two lengths don't match.
try:
    _ = da_work.assign_coords(hour_of_year=("time", hour_of_year_all))
    print("assign_coords OK")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

### 2b. Per-simulation reshape

Even if we fix `assign_coords`, the loop reshapes `values[:usable]` to `(n_years, 8760)` — that assumes every consecutive 8760-block is one calendar year aligned to Jan 1. If the head/tail are partial years, the reshape will silently misalign hour-of-year positions across years.

In [ ]:
# Sanity: do consecutive 8760-blocks actually start on Jan 1, hour 0?
times_da = pd.DatetimeIndex(da_work.time.values)
block_starts = times_da[::8760]
print("block starts (first 5):")
for t in block_starts[:5]:
    print(" ", t)
print("block starts (last 3):")
for t in block_starts[-3:]:
    print(" ", t)
print()
print("all blocks start Jan 1 00:00?:",
      bool(((block_starts.month == 1) & (block_starts.day == 1) & (block_starts.hour == 0)).all()))

## 3. Quantile selection (per-simulation loop)

If section 2 passes, this is the rest of `persistence_get_top_hours`. Run for one simulation to see the math.

In [ ]:
# Pick the first simulation and walk through the quantile selection.
sim_idx = 0
subset_data = da_work.isel(simulation=sim_idx) if has_simulation else da_work

values = subset_data.values
n_total = len(values)
usable = (n_total // hours_per_year) * hours_per_year
year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

print("sim                :", subset_data.simulation.item() if has_simulation else None)
print("n_total            :", n_total)
print("usable             :", usable)
print("year_hour_matrix   :", year_hour_matrix.shape)

quantile_targets = np.nanquantile(year_hour_matrix, q, axis=0)
print("quantile_targets[:5]:", quantile_targets[:5])
print("any inf in matrix? :", np.isinf(year_hour_matrix).any())
print("any nan in matrix? :", np.isnan(year_hour_matrix).any())

In [ ]:
diffs = np.abs(year_hour_matrix - quantile_targets[np.newaxis, :])
closest_year_idx = np.nanargmin(diffs, axis=0)

usable_times = subset_data.time.values[:usable]
row_years = pd.DatetimeIndex(usable_times).year.values.reshape(-1, hours_per_year)[:, 0]
sel_year = row_years[closest_year_idx]

print("row_years (first/last):", row_years[:3], "...", row_years[-3:])
print("closest_year_idx range:", closest_year_idx.min(), "..", closest_year_idx.max())
print("sel_year unique         :", np.unique(sel_year, return_counts=True))

## 4. Downstream steps (only run after sections 1–3 succeed)

Once `set_top_hours()` works, the rest of the pipeline is:

In [ ]:
# xmy.set_top_hours()
# xmy.run_xmy_analysis()
# xmy.export_xmy_data()

## 5. Open question to resolve before fixing

The user wants `skip_last` to drop **only the last month** (the original `.loc[dict(year=..., month=...)] = np.inf` intent). But the downstream code requires `total_hours` to be a multiple of 8760 *and* aligned to Jan 1, hour 0. Those constraints conflict for the WL+local-time case.

Options to consider, with cost:

1. **Mask-only (faithful to original `np.inf` intent).** Keep all stamps; replace last-month values with sentinel that `np.nanquantile` ignores → use `np.nan`, not `np.inf`. Then handle the resulting partial-year row in the reshape (pad to 8760 with NaN). Cost: reshape logic must change.
2. **Drop last partial year (current band-aid).** Lose ~1 year. Cost: data, but no logic changes.
3. **Roll the calendar.** Don't shift `time` to local; keep UTC and rotate the per-day hour axis at output time. Cost: bigger refactor, touches plotting/EPW writer.

Use this notebook to compare option 1's reshape logic against the ground-truth values before committing the fix.